importing datasets mnli and anli

In [1]:
from datasets import load_dataset
import pandas as pd

mnli = load_dataset("nyu-mll/multi_nli")

anli = load_dataset("facebook/anli")

print("MNLI\n", mnli["train"][:10])
print("\n\n\nANLI\n", anli)

MNLI
 {'promptID': [31193, 101457, 134793, 37397, 50563, 110116, 42487, 1069, 23505, 60529], 'pairID': ['31193n', '101457e', '134793e', '37397e', '50563n', '110116e', '42487n', '1069e', '23505c', '60529c'], 'premise': ['Conceptually cream skimming has two basic dimensions - product and geography.', 'you know during the season and i guess at at your level uh you lose them to the next level if if they decide to recall the the parent team the Braves decide to call to recall a guy from triple A then a double A guy goes up to replace him and a single A guy goes up to replace him', 'One of our number will carry out your instructions minutely.', 'How do you know? All this is their information again.', "yeah i tell you what though if you go price some of those tennis shoes i can see why now you know they're getting up in the hundred dollar range", "my walkman broke so i'm upset now i just have to turn the stereo up real loud", 'But a few Christian mosaics survive above the apse is the Virgin w

Testing quality and composition of mnli

In [2]:
labels_name = mnli["train"].features["label"].names
print(labels_name)

df_mnli_train = pd.DataFrame(mnli["train"])
distribution = df_mnli_train["label"].value_counts(normalize=True)
print(distribution.to_string())

index = 1

premise = mnli['train']['premise'][index]
hypothesis = mnli['train']['hypothesis'][index]
id_label = mnli['train']['label'][index]

print(f"Premise:    {premise}")
print(f"Hypothesis: {hypothesis}")
print(f"Label: {id_label} -> {labels_name[id_label]}")

['entailment', 'neutral', 'contradiction']
label
2    0.333339
1    0.333332
0    0.333329
Premise:    you know during the season and i guess at at your level uh you lose them to the next level if if they decide to recall the the parent team the Braves decide to call to recall a guy from triple A then a double A guy goes up to replace him and a single A guy goes up to replace him
Hypothesis: You lose the things to the following level if the people recall.
Label: 0 -> entailment


importing tokenizer for the models

also checking if I can cut some sequences without losing information (max_length=128)

In [3]:
from transformers import AutoTokenizer
import numpy as np

tok_distil = AutoTokenizer.from_pretrained("distilbert-base-uncased")
tok_deberta = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

def compute_lengths(batch):
    len_distil = [len(x) for x in tok_distil(batch["premise"], batch["hypothesis"])["input_ids"]]
    len_deberta = [len(x) for x in tok_deberta(batch["premise"], batch["hypothesis"])["input_ids"]]
    return {"len_distil": len_distil, "len_deberta": len_deberta}

mnli_lengths = mnli["train"].map(compute_lengths, batched=True, batch_size=1000)

lengths_d = np.array(mnli_lengths["len_distil"])
lengths_deb = np.array(mnli_lengths["len_deberta"])

print("\n--- DistilBERT (WordPiece) ---")
print(f"mean length:           {lengths_d.mean():.1f} token")
print(f"95° percentile:        {np.percentile(lengths_d, 95):.0f} token")
print(f"99° percentile:        {np.percentile(lengths_d, 99):.0f} token")
print(f"% over 128 token:     {(lengths_d > 128).mean() * 100:.2f}%")

print("\n--- DeBERTa-v3 (SentencePiece) ---")
print(f"mean length:           {lengths_deb.mean():.1f} token")
print(f"95° percentile:        {np.percentile(lengths_deb, 95):.0f} token")
print(f"99° percentile:        {np.percentile(lengths_deb, 99):.0f} token")
print(f"% over 128 token:     {(lengths_deb > 128).mean() * 100:.2f}%")


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Map:   0%|          | 0/392702 [00:00<?, ? examples/s]


--- DistilBERT (WordPiece) ---
mean length:           39.9 token
95° percentile:        73 token
99° percentile:        99 token
% over 128 token:     0.30%

--- DeBERTa-v3 (SentencePiece) ---
mean length:           38.8 token
95° percentile:        70 token
99° percentile:        96 token
% over 128 token:     0.27%
